# Real Estate Regression & Comprehensive Model Evaluation

In this notebook, we predict the exact property `Price` (continuous variable). We will evaluate 6 regression models:
1. Linear Regression
2. Support Vector Regressor (RBF Kernel)
3. Support Vector Regressor (Linear Kernel)
4. K-Nearest Neighbors Regressor (Distance-based regression)
5. Random Forest Regressor
6. XGBoost Regressor

For each model, we will calculate **MAE**, **RMSE**, and **R-squared ($R^2$)**. We will visualize their performance using **Actual vs. Predicted plots** and **Residual (Error) plots**, and finally rank them based on their $R^2$ score (which acts as regression accuracy).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

# Import Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

sns.set_theme(style="whitegrid", palette="muted")

# --- 1. LOAD & CLEAN DATA ---
df = pd.read_csv('vietnam_housing_dataset.csv')
df = df.dropna(subset=['Price'])

numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.drop('Price')
categorical_cols = df.select_dtypes(include=['object']).columns

for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')

# --- 2. EXTRACT DISTRICT ---
def extract_district(address):
    if pd.isna(address) or address == 'Unknown': return 'Unknown'
    parts = [part.strip() for part in str(address).split(',')]
    return parts[-2] if len(parts) >= 2 else 'Unknown'

df['District'] = df['Address'].apply(extract_district)

# --- 3. PREPARE DATA & ENCODE ---
X = df.drop(['Price', 'Address'], axis=1, errors='ignore') 
y = df['Price']
X_encoded = pd.get_dummies(X, drop_first=True)

# 4. Transform Target & Train-Test Split
y_log = np.log1p(df['Price'])
X_train, X_test, y_train_log, y_test_log = train_test_split(X_encoded, y_log, test_size=0.2, random_state=42)

# 5. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Global list to store results for final ranking
model_results = []

C:\Users\Admin\AppData\Local\Temp\ipykernel_22368\2918159096.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


## 1. Linear Regression

In [ ]:
print("--- Tuning Linear Regression ---")
lr = LinearRegression()
param_grid_lr = {
    'fit_intercept': [True, False]
}

grid_lr = GridSearchCV(lr, param_grid_lr, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_lr.fit(X_train_scaled, y_train_log)

print(f"Best Parameters: {grid_lr.best_params_}")
best_lr = grid_lr.best_estimator_

# Save Model
joblib.dump(best_lr, 'linear_regression.pkl')

# Predict and Inverse Transform
y_pred_log = best_lr.predict(X_test_scaled)
y_pred = np.expm1(y_pred_log)
y_te = np.expm1(y_test_log)

# Calculate Metrics
r2 = r2_score(y_te, y_pred)
mae = mean_absolute_error(y_te, y_pred)
mse = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared: {r2:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")

# 100 - MAPE Accuracy (%)
mape = mean_absolute_percentage_error(y_te, y_pred)
accuracy_mape = max(0, (1 - mape) * 100)
print(f"MAPE Accuracy: {accuracy_mape:.4f}")
model_results.append({'Model': 'Linear Regression', 'R-squared': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE Accuracy': accuracy_mape})

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
min_val, max_val = y_te.min() * 0.9, y_te.max() * 1.1

# Left: Actual vs Predicted as a density hexbin (thay cho scatter thường)
# — với ~6000 điểm test, scatter dễ bị chồng lấn (overplotting) nên khó thấy model nào
# dự đoán tập trung/lệch nhiều hơn. Hexbin tô màu theo mật độ điểm, giúp thấy rõ VÙNG
# model dự đoán tốt (đậm màu, sát đường đỏ) và vùng dự đoán kém (thưa hoặc lệch xa đường đỏ).
hb = axes[0].hexbin(y_te, y_pred, gridsize=40, cmap='Blues', mincnt=1,
                     extent=[min_val, max_val, min_val, max_val])
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
axes[0].set_title("Linear Regression: Actual vs Predicted (Density)")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
plt.colorbar(hb, ax=axes[0], label='Score')

# Right: Phân phối residual dạng histogram + KDE (thay cho residual scatter)
# — scatter residual khó so sánh "độ rộng sai số" giữa các model bằng mắt thường khi
# xem lần lượt từng cell. Histogram + KDE cho thấy ngay: đỉnh càng cao & hẹp quanh 0 =
# model càng chính xác; đuôi càng dài = model càng hay dự đoán sai lệch lớn.
residuals = y_te - y_pred
sns.histplot(residuals, kde=True, bins=40, ax=axes[1], color='darkorange')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title("Linear Regression: Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted, Billion VND)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 2. Support Vector Regressor (RBF Kernel)

In [ ]:
print("--- Tuning SVR (RBF Kernel) ---")
svr_rbf = SVR(kernel='rbf')
param_grid_rbf = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.1, 1] 
}

# 1. Take a 20% slice of the training data for FAST tuning
X_tune_rbf, _, y_tune_rbf, _ = train_test_split(X_train_scaled, y_train_log, train_size=0.20, random_state=42)

# 2. Run GridSearch ONLY on the 20% slice
grid_rbf = GridSearchCV(svr_rbf, param_grid_rbf, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_rbf.fit(X_tune_rbf, y_tune_rbf)

print(f"Best Parameters: {grid_rbf.best_params_}")

# 3. Train the winning model on the FULL 100% training data
best_rbf = grid_rbf.best_estimator_
best_rbf.fit(X_train_scaled, y_train_log)

# Save Model
joblib.dump(best_rbf, 'svr_rbf.pkl')

# Predict and Inverse Transform
y_pred_log = best_rbf.predict(X_test_scaled)
y_pred = np.expm1(y_pred_log)
y_te = np.expm1(y_test_log)

# Calculate Metrics
r2 = r2_score(y_te, y_pred)
mae = mean_absolute_error(y_te, y_pred)
mse = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared: {r2:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")

# 100 - MAPE Accuracy (%)
mape = mean_absolute_percentage_error(y_te, y_pred)
accuracy_mape = max(0, (1 - mape) * 100)
print(f"MAPE Accuracy: {accuracy_mape:.4f}")

model_results.append({'Model': 'SVR (RBF)', 'R-squared': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE Accuracy': accuracy_mape})

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
min_val, max_val = y_te.min() * 0.9, y_te.max() * 1.1

# Left: Actual vs Predicted as a density hexbin (thay cho scatter thường)
# — với ~6000 điểm test, scatter dễ bị chồng lấn (overplotting) nên khó thấy model nào
# dự đoán tập trung/lệch nhiều hơn. Hexbin tô màu theo mật độ điểm, giúp thấy rõ VÙNG
# model dự đoán tốt (đậm màu, sát đường đỏ) và vùng dự đoán kém (thưa hoặc lệch xa đường đỏ).
hb = axes[0].hexbin(y_te, y_pred, gridsize=40, cmap='Blues', mincnt=1,
                     extent=[min_val, max_val, min_val, max_val])
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
axes[0].set_title("SVR (RBF): Actual vs Predicted (Density)")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
plt.colorbar(hb, ax=axes[0], label='Score')

# Right: Phân phối residual dạng histogram + KDE (thay cho residual scatter)
# — scatter residual khó so sánh "độ rộng sai số" giữa các model bằng mắt thường khi
# xem lần lượt từng cell. Histogram + KDE cho thấy ngay: đỉnh càng cao & hẹp quanh 0 =
# model càng chính xác; đuôi càng dài = model càng hay dự đoán sai lệch lớn.
residuals = y_te - y_pred
sns.histplot(residuals, kde=True, bins=40, ax=axes[1], color='darkorange')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title("SVR (RBF): Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted, Billion VND)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 3. Support Vector Regressor (Linear Kernel)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import LinearSVR
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

print("--- Fast Tuning Linear SVR (GridSearchCV) ---")

# 1. LinearSVR with dual=False (optimal for datasets where n_samples > n_features)
svr_linear = LinearSVR(
    dual=False, 
    loss='squared_epsilon_insensitive', 
    random_state=42, 
    max_iter=2000
)

param_grid_linear = {
    'C': [0.01, 0.1, 1, 10, 100],
    'epsilon': [0.0, 0.1, 0.2]  # Tuning epsilon often improves precision
}

# 2. Standard GridSearchCV
grid_linear = GridSearchCV(
    estimator=svr_linear,
    param_grid=param_grid_linear,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# Fit on training data
grid_linear.fit(X_train_scaled, y_train_log)

print(f"Best Parameters: {grid_linear.best_params_}")

# 3. Extract best model (GridSearchCV automatically refits on 100% of X_train_scaled)
best_linear = grid_linear.best_estimator_

# Save Model
joblib.dump(best_linear, 'svr_linear.pkl')

# Predict and Inverse Transform
y_pred_log = best_linear.predict(X_test_scaled)
y_pred = np.expm1(y_pred_log)
y_te = np.expm1(y_test_log)

# Calculate Metrics
r2 = r2_score(y_te, y_pred)
mae = mean_absolute_error(y_te, y_pred)
mse = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(y_te, y_pred)
accuracy_mape = max(0, (1 - mape) * 100)

print(f"R-squared: {r2:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")
print(f"MAPE Accuracy: {accuracy_mape:.4f}%")

# Append to results
model_results.append({
    'Model': 'SVR (Linear)',
    'R-squared': r2,
    'MAE': mae,
    'MSE': mse,
    'RMSE': rmse,
    'MAPE Accuracy': accuracy_mape
})

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
min_val, max_val = y_te.min() * 0.9, y_te.max() * 1.1

# Left: Actual vs Predicted as a density hexbin (thay cho scatter thường)
# — với ~6000 điểm test, scatter dễ bị chồng lấn (overplotting) nên khó thấy model nào
# dự đoán tập trung/lệch nhiều hơn. Hexbin tô màu theo mật độ điểm, giúp thấy rõ VÙNG
# model dự đoán tốt (đậm màu, sát đường đỏ) và vùng dự đoán kém (thưa hoặc lệch xa đường đỏ).
hb = axes[0].hexbin(y_te, y_pred, gridsize=40, cmap='Blues', mincnt=1,
                     extent=[min_val, max_val, min_val, max_val])
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
axes[0].set_title("SVR (Linear): Actual vs Predicted (Density)")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
plt.colorbar(hb, ax=axes[0], label='Score')

# Right: Phân phối residual dạng histogram + KDE (thay cho residual scatter)
# — scatter residual khó so sánh "độ rộng sai số" giữa các model bằng mắt thường khi
# xem lần lượt từng cell. Histogram + KDE cho thấy ngay: đỉnh càng cao & hẹp quanh 0 =
# model càng chính xác; đuôi càng dài = model càng hay dự đoán sai lệch lớn.
residuals = y_te - y_pred
sns.histplot(residuals, kde=True, bins=40, ax=axes[1], color='darkorange')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title("SVR (Linear): Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted, Billion VND)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 4. K-Nearest Neighbors Regressor

In [ ]:
print("--- Tuning KNN Regressor ---")
knn = KNeighborsRegressor()
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance']
}

grid_knn = GridSearchCV(knn, param_grid_knn, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_knn.fit(X_train_scaled, y_train_log)

print(f"Best Parameters: {grid_knn.best_params_}")
best_knn = grid_knn.best_estimator_

# Save Model
joblib.dump(best_knn, 'knn.pkl')

# Predict and Inverse Transform
y_pred_log = best_knn.predict(X_test_scaled)
y_pred = np.expm1(y_pred_log)
y_te = np.expm1(y_test_log)

# Calculate Metrics
r2 = r2_score(y_te, y_pred)
mae = mean_absolute_error(y_te, y_pred)
mse = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared: {r2:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")

# 100 - MAPE Accuracy (%)
mape = mean_absolute_percentage_error(y_te, y_pred)
accuracy_mape = max(0, (1 - mape) * 100)
print(f"MAPE Accuracy: {accuracy_mape:.4f}")
model_results.append({'Model': 'KNN', 'R-squared': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE Accuracy': accuracy_mape})

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
min_val, max_val = y_te.min() * 0.9, y_te.max() * 1.1

# Left: Actual vs Predicted as a density hexbin (thay cho scatter thường)
# — với ~6000 điểm test, scatter dễ bị chồng lấn (overplotting) nên khó thấy model nào
# dự đoán tập trung/lệch nhiều hơn. Hexbin tô màu theo mật độ điểm, giúp thấy rõ VÙNG
# model dự đoán tốt (đậm màu, sát đường đỏ) và vùng dự đoán kém (thưa hoặc lệch xa đường đỏ).
hb = axes[0].hexbin(y_te, y_pred, gridsize=40, cmap='Blues', mincnt=1,
                     extent=[min_val, max_val, min_val, max_val])
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
axes[0].set_title("KNN: Actual vs Predicted (Density)")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
plt.colorbar(hb, ax=axes[0], label='Score')

# Right: Phân phối residual dạng histogram + KDE (thay cho residual scatter)
# — scatter residual khó so sánh "độ rộng sai số" giữa các model bằng mắt thường khi
# xem lần lượt từng cell. Histogram + KDE cho thấy ngay: đỉnh càng cao & hẹp quanh 0 =
# model càng chính xác; đuôi càng dài = model càng hay dự đoán sai lệch lớn.
residuals = y_te - y_pred
sns.histplot(residuals, kde=True, bins=40, ax=axes[1], color='darkorange')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title("KNN: Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted, Billion VND)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 5. Random Forest Regressor

In [ ]:
print("--- Tuning Random Forest ---")
rf = RandomForestRegressor(random_state=42)
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_rf.fit(X_train_scaled, y_train_log)

print(f"Best Parameters: {grid_rf.best_params_}")
best_rf = grid_rf.best_estimator_

# Save Model
joblib.dump(best_rf, 'random_forest.pkl')

# Predict and Inverse Transform
y_pred_log = best_rf.predict(X_test_scaled)
y_pred = np.expm1(y_pred_log)
y_te = np.expm1(y_test_log)

# Calculate Metrics
r2 = r2_score(y_te, y_pred)
mae = mean_absolute_error(y_te, y_pred)
mse = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared: {r2:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")

# 100 - MAPE Accuracy (%)
mape = mean_absolute_percentage_error(y_te, y_pred)
accuracy_mape = max(0, (1 - mape) * 100)
print(f"MAPE Accuracy: {accuracy_mape:.4f}")
model_results.append({'Model': 'Random Forest', 'R-squared': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE Accuracy': accuracy_mape})

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
min_val, max_val = y_te.min() * 0.9, y_te.max() * 1.1

# Left: Actual vs Predicted as a density hexbin (thay cho scatter thường)
# — với ~6000 điểm test, scatter dễ bị chồng lấn (overplotting) nên khó thấy model nào
# dự đoán tập trung/lệch nhiều hơn. Hexbin tô màu theo mật độ điểm, giúp thấy rõ VÙNG
# model dự đoán tốt (đậm màu, sát đường đỏ) và vùng dự đoán kém (thưa hoặc lệch xa đường đỏ).
hb = axes[0].hexbin(y_te, y_pred, gridsize=40, cmap='Blues', mincnt=1,
                     extent=[min_val, max_val, min_val, max_val])
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
axes[0].set_title("Random Forest: Actual vs Predicted (Density)")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
plt.colorbar(hb, ax=axes[0], label='Score')

# Right: Phân phối residual dạng histogram + KDE (thay cho residual scatter)
# — scatter residual khó so sánh "độ rộng sai số" giữa các model bằng mắt thường khi
# xem lần lượt từng cell. Histogram + KDE cho thấy ngay: đỉnh càng cao & hẹp quanh 0 =
# model càng chính xác; đuôi càng dài = model càng hay dự đoán sai lệch lớn.
residuals = y_te - y_pred
sns.histplot(residuals, kde=True, bins=40, ax=axes[1], color='darkorange')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title("Random Forest: Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted, Billion VND)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 6. XGBoost Regressor

In [ ]:
print("--- Tuning XGBoost ---")
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
param_grid_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

grid_xgb = GridSearchCV(xgb, param_grid_xgb, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_xgb.fit(X_train_scaled, y_train_log)

print(f"Best Parameters: {grid_xgb.best_params_}")
best_xgb = grid_xgb.best_estimator_

# Save Model
joblib.dump(best_xgb, 'xgboost.pkl')

# Predict and Inverse Transform
y_pred_log = best_xgb.predict(X_test_scaled)
y_pred = np.expm1(y_pred_log)
y_te = np.expm1(y_test_log)

# Calculate Metrics
r2 = r2_score(y_te, y_pred)
mae = mean_absolute_error(y_te, y_pred)
mse = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared: {r2:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")

# 100 - MAPE Accuracy (%)
mape = mean_absolute_percentage_error(y_te, y_pred)
accuracy_mape = max(0, (1 - mape) * 100)
print(f"MAPE Accuracy: {accuracy_mape:.4f}")
model_results.append({'Model': 'XGBoost', 'R-squared': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE Accuracy': accuracy_mape})

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
min_val, max_val = y_te.min() * 0.9, y_te.max() * 1.1

# Left: Actual vs Predicted as a density hexbin (thay cho scatter thường)
# — với ~6000 điểm test, scatter dễ bị chồng lấn (overplotting) nên khó thấy model nào
# dự đoán tập trung/lệch nhiều hơn. Hexbin tô màu theo mật độ điểm, giúp thấy rõ VÙNG
# model dự đoán tốt (đậm màu, sát đường đỏ) và vùng dự đoán kém (thưa hoặc lệch xa đường đỏ).
hb = axes[0].hexbin(y_te, y_pred, gridsize=40, cmap='Blues', mincnt=1,
                     extent=[min_val, max_val, min_val, max_val])
axes[0].plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
axes[0].set_title("XGBoost: Actual vs Predicted (Density)")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
plt.colorbar(hb, ax=axes[0], label='Score')

# Right: Phân phối residual dạng histogram + KDE (thay cho residual scatter)
# — scatter residual khó so sánh "độ rộng sai số" giữa các model bằng mắt thường khi
# xem lần lượt từng cell. Histogram + KDE cho thấy ngay: đỉnh càng cao & hẹp quanh 0 =
# model càng chính xác; đuôi càng dài = model càng hay dự đoán sai lệch lớn.
residuals = y_te - y_pred
sns.histplot(residuals, kde=True, bins=40, ax=axes[1], color='darkorange')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title("XGBoost: Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted, Billion VND)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


## 7. Final Model Ranking
Let's combine all the stored results into a single table and visualize the rankings based on their $R^2$ scores.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Convert to DataFrame
results_df = pd.DataFrame(model_results)

# Sort values by R-squared descending
results_df = results_df.sort_values(by='R-squared', ascending=False).reset_index(drop=True)

# Display Table with MAPE Accuracy formatting
print("--- Final Regression Model Rankings ---")
display(results_df.style.format({
    'R-squared': "{:.4f}",
    'MAE': "{:.4f}",
    'MSE': "{:.4f}",
    'RMSE': "{:.4f}",
    'MAPE Accuracy': "{:.2f}%"
}))

# Plot Bar Chart
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x='R-squared', y='Model', hue='Model', palette='viridis', legend=False)

plt.title('Model Performance Ranking (Based on R-squared)', fontsize=14, fontweight='bold')
plt.xlabel('R-squared ($R^2$) Score')
plt.ylabel('Machine Learning Model')

# Adjust limits and add labels (R-squared + MAPE Accuracy)
plt.xlim(min(0, results_df['R-squared'].min() * 1.1), 1.15) 

for index, row in results_df.iterrows():
    r2_val = row['R-squared']
    mape_acc = row['MAPE Accuracy']
    x_pos = r2_val + 0.01 if r2_val > 0 else 0.01 
    plt.text(x_pos, index, f"{r2_val:.4f} | Acc: {mape_acc:.2f}%", va='center')

plt.tight_layout()
plt.show()